In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Exercise 3: Camera Models and Distortion

**3D Computer Vision — Summer Semester 2026**

**Prof. Dr. Daniel Cremers, TU Munich**

---

In this notebook you will implement the pinhole camera model, the FOV/ATAN
distortion model, and a line-based distortion estimator (division model).
You will also see why all of this matters: a depth map plus a calibration
turns a flat image into a 3D point cloud, and a single line equation lets
us undo lens distortion without a calibration target.

| Part | Topic | What you'll see |
|------|-------|-----------------|
| I    | Pinhole project / backproject | Image + depth → 3D point cloud |
| II   | FOV/ATAN model | Undistortion of a fisheye KITTI frame at different output FOVs |
| III  | Line-based calibration (division model) | Estimate $k$ from 7 collinear points |

## Part I: Pinhole Projection and Backprojection

The pinhole camera maps a 3D point $\tilde{\mathbf X} = (X, Y, Z)^\top$
expressed in camera coordinates to a pixel $(u, v)$ via the intrinsic matrix
$K$:

$$\lambda \begin{pmatrix} u \\ v \\ 1 \end{pmatrix}
\;=\; K \begin{pmatrix} X \\ Y \\ Z \end{pmatrix},
\qquad
K = \begin{pmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{pmatrix}.$$

Reading off the third row gives $\lambda = Z$, so $u = f_x X / Z + c_x$ and
$v = f_y Y / Z + c_y$. Backprojection is the reverse: given a pixel and a
depth value, recover the 3D point. Since $\lambda \mathbf{x} = K \tilde{\mathbf X}$,

$$\tilde{\mathbf X} \;=\; Z \cdot K^{-1} \begin{pmatrix} u \\ v \\ 1 \end{pmatrix}.$$

**Implement** `pinhole_project` and `pinhole_backproject` below. You will
then use `pinhole_backproject` inside `image_to_pointcloud` to turn a full
RGB image plus its depth map into a coloured 3D point cloud.

In [ ]:
def pinhole_project(K, X):
    """Project 3D camera-frame points to pixels.

    Args:
        K: Intrinsic matrix, shape (3, 3).
        X: 3D points in camera frame, shape (N, 3).

    Returns:
        uv: Pixel coordinates, shape (N, 2).
    """
    x = X[:, 0] / X[:, 2]
    y = X[:, 1] / X[:, 2]
    u = K[0, 0] * x + K[0, 2]
    v = K[1, 1] * y + K[1, 2]
    return np.column_stack([u, v])

def pinhole_backproject(K, uv, depth):
    """Backproject pixels to 3D given per-pixel depth.

    Args:
        K: Intrinsic matrix, shape (3, 3).
        uv: Pixel coordinates, shape (N, 2).
        depth: Depth (Z value) at each pixel, shape (N,).

    Returns:
        X: 3D points in camera frame, shape (N, 3).
    """
    u, v = uv[:, 0], uv[:, 1]
    x = (u - K[0, 2]) / K[0, 0]
    y = (v - K[1, 2]) / K[1, 1]
    X = np.column_stack([x * depth, y * depth, depth])
    return X

In [ ]:
# ── Tests for Part I ──
_K = np.array([[500.0, 0.0, 320.0],
               [0.0, 500.0, 240.0],
               [0.0, 0.0, 1.0]])
_X = np.array([[0.1, 0.05, 1.0], [-0.2, 0.3, 2.5], [0.0, 0.0, 5.0]])
_uv = pinhole_project(_K, _X)
_X_back = pinhole_backproject(_K, _uv, _X[:, 2])
assert np.allclose(_X, _X_back, atol=1e-9), "project/backproject round-trip failed"
print("Part I: project/backproject round-trip OK")

### Build a point cloud from image + depth

The image `tum.jpg` is a single RGB frame; the file
`tum_calibration_depth.npz` provides a per-pixel depth map and the
corresponding intrinsics. Backprojecting every pixel using its depth
yields a coloured 3D point cloud --- turning a 2D image into a scene
we can rotate.

In [ ]:
_data = np.load("tum_calibration_depth.npz")
depth_tum = _data["depth"][0]            # (350, 504)
K_tum = _data["intrinsics"][0]           # (3, 3)
H_d, W_d = depth_tum.shape

_img = Image.open("tum.jpg").convert("RGB").resize((W_d, H_d), Image.BILINEAR)
rgb_tum = np.asarray(_img, dtype=np.float32) / 255.0

In [ ]:
_fig, _ax = plt.subplots(1, 2, figsize=(12, 4))
_ax[0].imshow(rgb_tum)
_ax[0].set_title("RGB frame (resized to depth resolution)")
_ax[0].axis("off")
_im = _ax[1].imshow(depth_tum, cmap="turbo")
_ax[1].set_title("Depth")
_ax[1].axis("off")
plt.colorbar(_im, ax=_ax[1], fraction=0.04, label="depth [m]")
plt.tight_layout()
plt.show()

In [ ]:
def image_to_pointcloud(K, rgb, depth):
    """Backproject every pixel of an image to a 3D point cloud.

    Args:
        K: Intrinsic matrix, shape (3, 3).
        rgb: RGB image, shape (H, W, 3), values in [0, 1].
        depth: Per-pixel depth, shape (H, W).

    Returns:
        points: 3D points in camera frame, shape (H*W, 3).
        colors: Per-point RGB, shape (H*W, 3).
    """
    H, W = depth.shape
    v, u = np.mgrid[0:H, 0:W]
    uv = np.column_stack([u.ravel(), v.ravel()])
    points = pinhole_backproject(K, uv, depth.ravel())
    colors = rgb.reshape(-1, 3)
    return points, colors

In [ ]:
points_3d, colors_3d = image_to_pointcloud(K_tum, rgb_tum, depth_tum)

In [ ]:
# Subsample for interactive plotting
import plotly.graph_objects as go
_step = 4
_pts = points_3d[::_step]
_col = colors_3d[::_step]
_color_str = [f"rgb({int(255*r)},{int(255*g)},{int(255*b)})" for r, g, b in _col]
fig_cloud = go.Figure(data=[go.Scatter3d(
    x=_pts[:, 0], y=_pts[:, 1], z=_pts[:, 2],
    mode="markers",
    marker=dict(size=1.5, color=_color_str, opacity=0.9),
)])
fig_cloud.update_layout(
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode="data",
        camera=dict(eye=dict(x=0.0, y=-0.2, z=-1.8), up=dict(x=0, y=-1, z=0)),
    ),
    margin=dict(l=0, r=0, b=0, t=0),
    height=600,
)
fig_cloud

## Part II: FOV / ATAN Camera Model

Real lenses do not project through a perfect pinhole. The
**FOV / ATAN model** (Devernay & Faugeras) relates the radius
$r_u = \|\pi(\tilde{\mathbf X})\|$ a perfect pinhole would produce to the
radius $r_d = \|\pi_d(\tilde{\mathbf X})\|$ actually measured on the sensor
by

$$r_d \;=\; \frac{1}{\omega} \arctan\!\big(2\, r_u \tan(\omega/2)\big).$$

Both directions keep the point on the same ray from the optical center, so the
full transformation rescales the normalized vector by $r_d / r_u$ (forward) or
$r_u / r_d$ (inverse).

**Implement** `fov_distort` (forward) using the formula above, and
`fov_undistort` (inverse) using the **closed-form inverse you derived in
Theory Exercise 3, Problem 4(b)**.

In [ ]:
def fov_distort(pi_u, omega):
    """Forward FOV/ATAN model: undistorted → distorted normalized coords.

    Args:
        pi_u: Undistorted normalized coords, shape (N, 2).
        omega: FOV parameter (radians).

    Returns:
        pi_d: Distorted normalized coords, shape (N, 2).
    """
    r_u = np.linalg.norm(pi_u, axis=1)
    # Use a small-radius limit to avoid 0/0 at the optical center
    r_d = np.where(
        r_u > 1e-9,
        np.arctan(2.0 * r_u * np.tan(omega / 2.0)) / omega,
        r_u,
    )
    scale = np.where(r_u > 1e-9, r_d / np.maximum(r_u, 1e-9), 1.0)
    return pi_u * scale[:, None]

def fov_undistort(pi_d, omega):
    """Inverse FOV/ATAN model: distorted → undistorted normalized coords.

    Args:
        pi_d: Distorted normalized coords, shape (N, 2).
        omega: FOV parameter (radians).

    Returns:
        pi_u: Undistorted normalized coords, shape (N, 2).
    """
    r_d = np.linalg.norm(pi_d, axis=1)
    r_u = np.where(
        r_d > 1e-9,
        np.tan(omega * r_d) / (2.0 * np.tan(omega / 2.0)),
        r_d,
    )
    scale = np.where(r_d > 1e-9, r_u / np.maximum(r_d, 1e-9), 1.0)
    return pi_d * scale[:, None]

In [ ]:
# ── Forward-backward stability test ──
_rng = np.random.default_rng(0)
_pi_u = _rng.uniform(-0.8, 0.8, size=(2000, 2))
_omega = 0.93
_pi_d = fov_distort(_pi_u, _omega)
_pi_u_back = fov_undistort(_pi_d, _omega)
_err = np.linalg.norm(_pi_u - _pi_u_back, axis=1).max()
assert _err < 1e-8, f"Forward-backward not stable, max err = {_err}"
print(f"Part II: forward-backward round-trip max error = {_err:.2e}")

### Build a pinhole from a desired output field of view

To rectify a fisheye image we need a *target* pinhole intrinsic matrix.
Given a desired horizontal field of view $\text{FOV}_x$ (in degrees) and an
output image size $(W, H)$, derive the focal length $f$ such that the image
spans exactly that angle. Place the principal point at the image center
$c_x = W/2,\; c_y = H/2$ and use the same focal length on both axes.

*Hint:* draw the triangle formed by the optical center, the principal
point, and the rightmost pixel column. The half-angle is $\text{FOV}_x/2$,
the opposite side is $W/2$, and the adjacent side is $f$.

**Implement** `pinhole_from_fov` below.

In [ ]:
def pinhole_from_fov(fov_x_deg, width, height):
    """Build a pinhole K matrix with principal point at the image center.

    Args:
        fov_x_deg: Desired horizontal FOV in degrees.
        width: Output image width.
        height: Output image height.

    Returns:
        K: Intrinsic matrix, shape (3, 3).
    """
    fov_x = np.deg2rad(fov_x_deg)
    f = (width / 2.0) / np.tan(fov_x / 2.0)
    return np.array([[f, 0.0, width / 2.0],
                     [0.0, f, height / 2.0],
                     [0.0, 0.0, 1.0]])

### Undistort the KITTI fisheye image

The KITTI frame `kitti.png` was captured by a wide-FOV camera modelled by
the FOV/ATAN distortion with parameters given below. We will resample it
through a virtual pinhole of our choosing using `cv2.remap`.

**Pipeline.** For each pixel $(u_p, v_p)$ in the *output* pinhole image:

1. normalize using $K_{\text{pin}}^{-1}$ → undistorted normalized coords;
2. apply `fov_distort` → distorted normalized coords;
3. apply $K_{\text{fisheye}}$ → source pixel coords;
4. sample the source image at those coords (bilinear).

Step 4 is what `cv2.remap` does for us. The mapping itself is what you
already implemented.

In [ ]:
img_kitti = np.asarray(Image.open("kitti.png").convert("RGB"))
line_pts = np.loadtxt("line_points.txt")    # (N, 2) distorted pixel coords

# Embedded FOV/ATAN parameters for kitti.png
fx_k, fy_k = 385.5633239746094, 385.4128723144531
cx_k, cy_k = 716.9444580078125, 705.7637329101562
omega_k = 0.9310157895088196
K_kitti = np.array([[fx_k, 0.0, cx_k],
                    [0.0, fy_k, cy_k],
                    [0.0, 0.0, 1.0]])

In [ ]:
import cv2

def undistort_fov_to_pinhole(img_src, K_src, omega, K_dst, out_w, out_h):
    """Remap a FOV/ATAN-distorted image through a virtual pinhole."""
    _v, _u = np.mgrid[0:out_h, 0:out_w].astype(np.float32)
    _uv_pin = np.column_stack([_u.ravel(), _v.ravel()])

    # Step 1: pinhole pixel → undistorted normalized
    _xn = (_uv_pin[:, 0] - K_dst[0, 2]) / K_dst[0, 0]
    _yn = (_uv_pin[:, 1] - K_dst[1, 2]) / K_dst[1, 1]
    _pi_u = np.column_stack([_xn, _yn])

    # Step 2: apply FOV distortion → distorted normalized
    _pi_d = fov_distort(_pi_u, omega)

    # Step 3: distorted normalized → source pixel
    _u_src = K_src[0, 0] * _pi_d[:, 0] + K_src[0, 2]
    _v_src = K_src[1, 1] * _pi_d[:, 1] + K_src[1, 2]

    map_x = _u_src.reshape(out_h, out_w).astype(np.float32)
    map_y = _v_src.reshape(out_h, out_w).astype(np.float32)
    return cv2.remap(img_src, map_x, map_y, cv2.INTER_LINEAR, borderValue=(0, 0, 0))

In [ ]:
fovs_deg = [60, 90, 120, 150]
out_w, out_h = 900, 900
_fig, _axes = plt.subplots(1, len(fovs_deg) + 1, figsize=(20, 5))
_axes[0].imshow(img_kitti)
_axes[0].set_title("Original (distorted)")
_axes[0].axis("off")
for _i, _fov in enumerate(fovs_deg):
    _K_pin = pinhole_from_fov(_fov, out_w, out_h)
    _out = undistort_fov_to_pinhole(img_kitti, K_kitti, omega_k, _K_pin, out_w, out_h)
    _axes[_i + 1].imshow(_out)
    _axes[_i + 1].set_title(f"Pinhole, FOV = {_fov}°")
    _axes[_i + 1].axis("off")
plt.tight_layout()
plt.show()

## Part III: Line-Based Distortion Estimation (Division Model)

We now estimate distortion without a calibration target. The **division
model** maps observed distorted normalized coordinates
$\pi_d = (x_d, y_d)^\top$ directly to undistorted ones:

$$x_u = \frac{x_d}{1 + k\,r_d^2},
\qquad y_u = \frac{y_d}{1 + k\,r_d^2},
\qquad r_d^2 = x_d^2 + y_d^2.$$

**Idea.** Points that lie on a straight 3D line should also lie on a
straight line in the undistorted image. Plugging the division model into
the collinearity condition for three observed points produces an equation
whose only unknown is $k$. Stacking such equations over many triplets
gives you a small least-squares problem.

The full derivation --- collinearity as a $3\times 3$ determinant, the
algebraic manipulation that makes it linear in $k$, and the resulting
closed-form estimate --- is **Theory Exercise 3, Problem 5**. Use what
you derive there to implement the estimator.

Since we only want $k$ (not full intrinsics), we use a placeholder camera
with the principal point at the image center and the focal length set to
the image width: $f = W,\; c_x = W/2,\; c_y = H/2$.

**Implement** `estimate_k_division` below.

In [ ]:
def estimate_k_division(uv_pixels, K):
    """Estimate the division-model coefficient k from N pixels on one line.

    Args:
        uv_pixels: Distorted pixel coords on a single line, shape (N, 2).
        K: Simple intrinsic matrix used to normalize coords, shape (3, 3).

    Returns:
        k_hat: Estimated division-model coefficient (scalar).
    """
    u = uv_pixels[:, 0]
    v = uv_pixels[:, 1]
    x_d = (u - K[0, 2]) / K[0, 0]
    y_d = (v - K[1, 2]) / K[1, 1]
    r2 = x_d ** 2 + y_d ** 2

    N = len(uv_pixels)
    A_list, B_list = [], []
    for i in range(N):
        for j in range(i + 1, N):
            for k in range(j + 1, N):
                M_one = np.array([
                    [x_d[i], y_d[i], 1.0],
                    [x_d[j], y_d[j], 1.0],
                    [x_d[k], y_d[k], 1.0],
                ])
                M_r2 = np.array([
                    [x_d[i], y_d[i], r2[i]],
                    [x_d[j], y_d[j], r2[j]],
                    [x_d[k], y_d[k], r2[k]],
                ])
                A_list.append(np.linalg.det(M_one))
                B_list.append(np.linalg.det(M_r2))
    A = np.array(A_list)
    B = np.array(B_list)
    return -np.sum(A * B) / np.sum(B * B)

In [ ]:
H_img, W_img = img_kitti.shape[:2]
K_simple = np.array([[W_img, 0.0, W_img / 2.0],
                     [0.0, W_img, H_img / 2.0],
                     [0.0, 0.0, 1.0]])
k_hat = estimate_k_division(line_pts, K_simple)
print(f"Estimated division-model coefficient: k = {k_hat:.4f}")

In [ ]:
import cv2 as _cv2

def undistort_division(img, K_src, k, K_dst, out_w, out_h):
    """Resample a division-model image through a virtual pinhole.

    For each output pinhole pixel (treated as undistorted) we find the source
    (distorted) pixel by inverting the division model. The division model
    ``x_u = x_d / (1 + k r_d^2)`` is undistortion; inverting it for r_d gives
        r_d = (1 - sqrt(1 - 4 k r_u^2)) / (2 k r_u)   (k != 0)
    which is one-to-one for the central well-behaved disk.
    """
    _v, _u = np.mgrid[0:out_h, 0:out_w].astype(np.float32)
    _x_u = (_u - K_dst[0, 2]) / K_dst[0, 0]
    _y_u = (_v - K_dst[1, 2]) / K_dst[1, 1]
    _r_u2 = _x_u ** 2 + _y_u ** 2
    if abs(k) < 1e-12:
        _scale = np.ones_like(_r_u2)
    else:
        _disc = np.clip(1.0 - 4.0 * k * _r_u2, 0.0, None)
        _r_u = np.sqrt(np.maximum(_r_u2, 1e-12))
        _r_d = (1.0 - np.sqrt(_disc)) / (2.0 * k * _r_u)
        _scale = np.where(_r_u2 > 1e-12, _r_d / _r_u, 1.0)
    _x_d = _x_u * _scale
    _y_d = _y_u * _scale
    _u_src = (K_src[0, 0] * _x_d + K_src[0, 2]).astype(np.float32)
    _v_src = (K_src[1, 1] * _y_d + K_src[1, 2]).astype(np.float32)
    return _cv2.remap(img, _u_src, _v_src, _cv2.INTER_LINEAR, borderValue=(0, 0, 0))

def line_pts_division_to_pinhole(pts_dist, K_src, k, K_dst):
    """Map distorted source-pixel coords to undistorted pinhole-output pixels."""
    _x_d = (pts_dist[:, 0] - K_src[0, 2]) / K_src[0, 0]
    _y_d = (pts_dist[:, 1] - K_src[1, 2]) / K_src[1, 1]
    _scale = 1.0 / (1.0 + k * (_x_d ** 2 + _y_d ** 2))
    _x_u = _x_d * _scale
    _y_u = _y_d * _scale
    return np.column_stack([
        K_dst[0, 0] * _x_u + K_dst[0, 2],
        K_dst[1, 1] * _y_u + K_dst[1, 2],
    ])

In [ ]:
out_w_div, out_h_div = 900, 900
# Output focal lengths chosen as fractions of the image width — the FOV/ATAN
# parameters are unknown for this estimator, so we just sweep zoom.
focal_scales = [0.25, 0.5, 0.6, 0.8]
_fig, _axes = plt.subplots(1, len(focal_scales) + 1, figsize=(22, 5))
_axes[0].imshow(img_kitti)
_axes[0].scatter(line_pts[:, 0], line_pts[:, 1], s=30, c="red",
                 edgecolors="white", linewidths=0.8)
_axes[0].plot(line_pts[:, 0], line_pts[:, 1], "r-", lw=0.8, alpha=0.5)
_axes[0].set_title("Original — points appear curved")
_axes[0].axis("off")
for _i, _s in enumerate(focal_scales):
    _f_out = _s * img_kitti.shape[1]
    _K_pin = np.array([[_f_out, 0.0, out_w_div / 2.0],
                       [0.0, _f_out, out_h_div / 2.0],
                       [0.0, 0.0, 1.0]])
    _out = undistort_division(img_kitti, K_simple, k_hat, _K_pin,
                              out_w_div, out_h_div)
    _pts_out = line_pts_division_to_pinhole(line_pts, K_simple, k_hat, _K_pin)
    _axes[_i + 1].imshow(_out)
    _axes[_i + 1].scatter(_pts_out[:, 0], _pts_out[:, 1], s=30, c="red",
                          edgecolors="white", linewidths=0.8)
    _axes[_i + 1].plot(_pts_out[:, 0], _pts_out[:, 1], "r-", lw=0.8, alpha=0.5)
    _all_x = np.concatenate([[0, out_w_div], _pts_out[:, 0]])
    _all_y = np.concatenate([[0, out_h_div], _pts_out[:, 1]])
    _pad = 30
    _axes[_i + 1].set_xlim(_all_x.min() - _pad, _all_x.max() + _pad)
    _axes[_i + 1].set_ylim(_all_y.max() + _pad, _all_y.min() - _pad)
    _axes[_i + 1].set_aspect("equal")
    _axes[_i + 1].set_title("Undistorted")
    _axes[_i + 1].axis("off")
plt.tight_layout()
plt.show()